# S03E05 — Save Them (Route Planning)

Wytyczenie optymalnej trasy dla posłańca do miasta Skolwin.
- Mapa 10x10 z rzeką, skałami, drzewami
- 10 porcji jedzenia, 10 jednostek paliwa
- Pojazdy: walk, horse, car, rocket
- Narzędzia odkrywane przez `/api/toolsearch`

In [ ]:
import requests, os, json
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("AI_DEVS_API_KEY")
VERIFY_URL = "https://hub.ag3nts.org/verify"
BASE = "https://hub.ag3nts.org"

def api(endpoint, query):
    r = requests.post(f"{BASE}{endpoint}", json={"apikey": API_KEY, "query": query})
    return r.json()

## 1. Odkryj narzędzia

In [ ]:
tools = api("/api/toolsearch", "map terrain vehicles rules notes instructions")
print(json.dumps(tools, indent=2))

## 2. Pobierz mapę, pojazdy i zasady

In [ ]:
# Mapa Skolwina
map_data = api("/api/maps", "Skolwin")
print("Mapa:")
print(map_data["text"])
grid = map_data["map"]

# Znajdz S i G
start = goal = None
for r, row in enumerate(grid):
    for c, cell in enumerate(row):
        if cell == "S": start = (r, c)
        if cell == "G": goal = (r, c)
print(f"\nStart: {start}, Goal: {goal}")

In [ ]:
# Pojazdy
vehicles = {}
for v in ["walk", "horse", "car", "rocket"]:
    data = api("/api/wehicles", v)
    vehicles[v] = data["consumption"]
    print(f"{v:8s}: fuel={data['consumption']['fuel']}, food={data['consumption']['food']}")

In [ ]:
# Zasady
for q in [
    "water rocks terrain types movement",
    "trees fuel burn consumption",
    "dismount walk water river crossing vehicles",
    "map orientation command format",
]:
    result = api("/api/books", q)
    for note in result.get("notes", []):
        print(f"--- {note['title']} ---")
        print(note["content"][:200])
        print()

## 3. Planowanie trasy (BFS z optymalizacją zasobów)

Zasady:
- `R` = skały (nie do przejścia)
- `W` = woda (tylko horse lub walk)
- `T` = drzewa (+0.2 paliwa dla pojazdów silnikowych)
- Rocket: fuel=1.0, food=0.1 (nie może na wodę)
- Walk: fuel=0, food=2.5 (może na wodę)
- Strategia: rocket do rzeki, dismount, walk przez wodę do celu

In [ ]:
from heapq import heappush, heappop

DIRS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
ROWS, COLS = len(grid), len(grid[0])


def solve(grid, start, goal, max_fuel=10.0, max_food=10.0):
    """BFS/Dijkstra: state = (row, col, mode) where mode is 'rocket' or 'walk'.
    Minimizes total moves. Returns command list."""
    # State: (total_moves, fuel_used, food_used, row, col, mode, commands)
    initial = (0, 0.0, 0.0, start[0], start[1], "rocket", ["rocket"])
    visited = set()
    queue = [initial]

    while queue:
        moves, fuel, food, r, c, mode, cmds = heappop(queue)

        if (r, c) == goal:
            return cmds, fuel, food

        state_key = (r, c, mode)
        if state_key in visited:
            continue
        visited.add(state_key)

        # Try dismount (if in vehicle)
        if mode == "rocket":
            new_state = (moves, fuel, food, r, c, "walk", cmds + ["dismount"])
            if (r, c, "walk") not in visited:
                heappush(queue, new_state)

        # Try moving in each direction
        for dir_name, (dr, dc) in DIRS.items():
            nr, nc = r + dr, c + dc
            if not (0 <= nr < ROWS and 0 <= nc < COLS):
                continue

            tile = grid[nr][nc]
            if tile == "R":  # rocks block all
                continue

            if tile == "W" and mode == "rocket":  # rocket can't cross water
                continue

            # Calculate resource cost
            if mode == "rocket":
                df = 1.0 + (0.2 if tile == "T" else 0.0)
                dd = 0.1
            else:  # walk
                df = 0.0
                dd = 2.5

            new_fuel = fuel + df
            new_food = food + dd

            if new_fuel > max_fuel or new_food > max_food:
                continue

            new_state = (moves + 1, new_fuel, new_food, nr, nc, mode, cmds + [dir_name])
            if (nr, nc, mode) not in visited:
                heappush(queue, new_state)

    return None, None, None


cmds, fuel, food = solve(grid, start, goal)
if cmds:
    print(f"Trasa znaleziona! {len(cmds)-1} krokow (+ nazwa pojazdu)")
    print(f"Paliwo: {fuel}/10, Jedzenie: {food:.1f}/10")
    print(f"Komendy: {cmds}")
else:
    print("Nie znaleziono trasy!")

## 4. Wyślij trasę do centrali

In [ ]:
resp = requests.post(VERIFY_URL, json={
    "apikey": API_KEY,
    "task": "savethem",
    "answer": cmds,
})
print(resp.json())